# M49 · segmentation polygon trên Colab
YOLO11s-seg → JSON M49. Pipeline đã viết, chưa chạy inference GPU thực tế.
Backend dùng Python 3.14; Colab là môi trường AI riêng. Không cài Torch/YOLO vào backend.

Tạo lượt Colab trong ứng dụng, mount Drive và điền mã lượt vào ROOT (1–20 ảnh).
Kết quả tự ghi vào Drive để backend nhận. Thử 3–5 ảnh pilot trước khi khóa cấu hình.
Mỗi instance phải là một vùng không có lỗ; gặp multipart/holes thì dừng, không bỏ dự đoán.
Polygon là xấp xỉ biên mask mô hình, không phải mask pixel gốc. Ghi rõ khi báo cáo.
Không train và không dùng ground truth để điều chỉnh dự đoán.

Tài liệu: https://docs.ultralytics.com/tasks/segment/ và https://docs.ultralytics.com/models/yolo11/


In [ ]:
%pip install ultralytics==8.3.203 pydantic==2.14.1


In [ ]:
import json, time, hashlib, subprocess, sys
from pathlib import Path
import torch, ultralytics
from PIL import Image
from ultralytics import YOLO

# M49_DRIVE_TRANSPORT
"""Shared by generated notebook cells; stdlib only, testable without Colab."""
import hashlib
import json
from pathlib import Path


def load_drive_job(root, task):
    root = Path(root)
    request = json.loads((root/'request.json').read_text(encoding='utf-8'))
    if request['task'] != task or root.name != request['job_id']:
        raise ValueError('Sai notebook hoặc sai thư mục lượt chạy')
    names = request['manifest']
    if not isinstance(names, list) or not 1 <= len(names) <= 20 or len(names) != len(set(names)):
        raise ValueError('Manifest không hợp lệ')
    for name in names:
        if not isinstance(name, str) or not name or '/' in name or '\\' in name or ':' in name or name in {'.', '..'}:
            raise ValueError('Tên mẫu không hợp lệ')
        path = root/'assets'/name
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != request['sha256'].get(name):
            raise ValueError(f'{name}: chờ Drive đồng bộ đủ dữ liệu rồi chạy lại')
    return request


def publish_drive_result(root, request, bundle, timing, environment):
    root = Path(root)
    if root.name != request['job_id'] or bundle['task'] != request['task']:
        raise ValueError('Sai lượt chạy hoặc tác vụ')
    if sorted(s['asset_name'] for s in bundle['samples']) != sorted(request['manifest']):
        raise ValueError('Kết quả thiếu/trùng/thừa mẫu')
    if (root/'result.ready.json').exists():
        raise ValueError('Lượt này đã xuất kết quả. Tạo lượt mới trong ứng dụng để chạy lại.')
    raw = json.dumps(bundle, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8')
    if len(raw) > 32*1024*1024:
        raise ValueError('Kết quả vượt 32 MiB')
    outputs = {'predictions.json': raw,
               'timing.json': json.dumps(timing, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8'),
               'environment.txt': environment.encode('utf-8')}
    for name, data in outputs.items():
        temporary = root/(name+'.tmp')
        temporary.write_bytes(data)
        temporary.replace(root/name)
    receipt = {'job_id': request['job_id'], 'sha256': hashlib.sha256(raw).hexdigest()}
    temporary = root/'result.ready.tmp'
    temporary.write_text(json.dumps(receipt), encoding='utf-8')
    temporary.replace(root/'result.ready.json')
    return receipt

from google.colab import drive
drive.mount('/content/drive')
# Copy the job ID displayed in the app. This is the Colab path, not the Windows path.
ROOT = Path('/content/drive/MyDrive/M49/PASTE_JOB_ID')
request = load_drive_job(ROOT, 'instance_segmentation')

ASSETS = ROOT / 'assets'
# M49_MODEL_SELECTION
model_spec = request.get('model', {})
CHECKPOINT = model_spec.get('checkpoint', 'yolo11s-seg.pt')
if model_spec.get('custom'):
    model_path = (ROOT.parent / CHECKPOINT).resolve()
    assert model_path.is_relative_to((ROOT.parent / 'models').resolve()), 'Checkpoint ngoài thư mục models'
    assert model_path.is_file(), 'Chờ Drive đồng bộ checkpoint'
    assert hashlib.sha256(model_path.read_bytes()).hexdigest() == model_spec['sha256'], 'Checkpoint chưa đồng bộ đủ'
    CHECKPOINT = str(model_path)

# Exact semantic aliases only; Cyclist/rider cannot be replaced by a person box.
aliases = {'pedestrian': 'person', 'Car': 'car', 'Pedestrian': 'person', 'Truck': 'truck', 'Tram': 'train'}
CLASS_MAP = {aliases.get(category, category): category for category in request['classes']}
assert len(CLASS_MAP) == len(request['classes']), 'Hai lớp dự án trùng một lớp model'

CONFIDENCE = model_spec.get('confidence', 0.25)  # Tune on pilot only, then freeze.
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def stamp():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.perf_counter()

names = request['manifest']
assert isinstance(names, list) and 1 <= len(names) <= 20
assert all(isinstance(n, str) and n and '/' not in n and chr(92) not in n and Path(n).name == n for n in names)
assert len(set(names)) == len(names), 'Tên ảnh trùng'
paths = [(ASSETS / name).resolve() for name in names]
assert all(p.parent == ASSETS.resolve() and p.is_file() for p in paths), 'Thiếu ảnh hoặc đường dẫn không hợp lệ'
assert CLASS_MAP and all(isinstance(v, str) and v.strip() for v in CLASS_MAP.values())
print('Số ảnh:', len(paths), 'Thiết bị:', DEVICE)


In [ ]:
# M49_POLYGON_VALIDATION: generated from backend schema; tested locally.
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, model_validator

class StrictModel(BaseModel):
    model_config = ConfigDict(extra='forbid', allow_inf_nan=False)

class Polygon(StrictModel):
    kind: Literal["polygon"]
    points: list[tuple[float, float]] = Field(min_length=3, max_length=1000)

    @model_validator(mode="after")
    def simple_polygon(self):
        points = self.points
        if len(set(points)) != len(points):
            raise ValueError('Polygon không được có đỉnh trùng; không lặp đỉnh đầu ở cuối')
        signed = sum(a[0]*b[1]-b[0]*a[1] for a, b in zip(points, points[1:]+points[:1]))
        if abs(signed) < 1e-9:
            raise ValueError('Polygon phải có diện tích dương')
        def cross(a, b, c):
            return (b[0]-a[0])*(c[1]-a[1])-(b[1]-a[1])*(c[0]-a[0])
        def on(a, b, p):
            return min(a[0],b[0]) <= p[0] <= max(a[0],b[0]) and min(a[1],b[1]) <= p[1] <= max(a[1],b[1])
        for i, b in enumerate(points):
            a, c = points[i-1], points[(i+1) % len(points)]
            if cross(a,b,c) == 0 and (b[0]-a[0])*(c[0]-b[0])+(b[1]-a[1])*(c[1]-b[1]) < 0:
                raise ValueError('Các cạnh kề của polygon không được chồng lên nhau')
        edges = list(zip(points, points[1:]+points[:1]))
        for i, (a, b) in enumerate(edges):
            for j in range(i+1, len(edges)):
                if j == i+1 or (i == 0 and j == len(edges)-1):
                    continue
                c, d = edges[j]
                ab_c, ab_d, cd_a, cd_b = cross(a,b,c), cross(a,b,d), cross(c,d,a), cross(c,d,b)
                if (ab_c*ab_d < 0 and cd_a*cd_b < 0) or any((value == 0 and on(u,v,p)) for value,u,v,p in
                        [(ab_c,a,b,c), (ab_d,a,b,d), (cd_a,c,d,a), (cd_b,c,d,b)]):
                    raise ValueError('Polygon không được tự giao hoặc tự chạm')
        return self

def validate_polygon(points, width, height):
    points = [list(p) for p in points]
    if points and points[0] == points[-1]:
        points.pop()
    polygon = Polygon(kind='polygon', points=points)
    if any(not (0 <= x <= width and 0 <= y <= height) for x,y in polygon.points):
        raise ValueError('Polygon nằm ngoài ảnh gốc')
    return polygon.model_dump(mode='json')['points']


In [ ]:
import cv2
import numpy as np

start = stamp()
model = YOLO(CHECKPOINT)
load_seconds = stamp() - start
assert model.task == 'segment', 'Cần checkpoint segmentation (-seg.pt)'
CLASS_MAP = {(category if category in model.names.values() else aliases.get(category, category)): category for category in request['classes']}
assert set(CLASS_MAP) <= set(model.names.values()), 'Lớp không có trong checkpoint'
class_ids = [i for i, name in model.names.items() if name in CLASS_MAP]
for path in paths:
    with Image.open(path) as image:
        assert image.width*image.height <= 4_000_000, 'Ảnh vượt ngân sách 4 triệu pixel'
kwargs = dict(imgsz=IMGSZ, conf=CONFIDENCE, classes=class_ids, device=DEVICE,
              max_det=80, retina_masks=True, verbose=False, save=False)
start = stamp()
model.predict(str(paths[0]), **kwargs)
warmup_seconds = stamp() - start
samples, per_image = [], []
for name, path in zip(names, paths):
    with Image.open(path) as image:
        width, height = image.size
    start = stamp()
    result = model.predict(str(path), **kwargs)[0]
    predict_seconds = stamp() - start
    start = stamp()
    objects = []
    boxes = result.boxes
    count = len(boxes) if boxes is not None else 0
    assert count < 80, f'{name}: chạm max_det; chọn pilot ít đối tượng hơn để tránh cắt nhãn'
    assert width*height*count <= 16_000_000, f'{name}: cần giảm mẫu để còn ngân sách chấm ground truth'
    if count:
        assert result.masks is not None and len(result.masks.data)==count
        assert tuple(result.masks.data.shape[1:])==(height,width), 'Mask phải có độ phân giải ảnh gốc'
        polygons = result.masks.xy
        assert len(polygons)==count
        classes, confidences = boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist()
        for index, (polygon, cls, conf) in enumerate(zip(polygons, classes, confidences)):
            mask = (result.masks.data[index].cpu().numpy()>0.5).astype(np.uint8)
            contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours)!=1 or hierarchy is None or hierarchy[0,0,2]!=-1:
                raise ValueError(f'{name}, instance {index}: mask rỗng, nhiều vùng hoặc có lỗ; M49 chưa hỗ trợ')
            points = validate_polygon(polygon.tolist(),width,height)
            objects.append(dict(id=f'ai-{index}',category=CLASS_MAP[model.names[int(cls)]],
                confidence=float(conf),geometry=dict(kind='polygon',points=points)))
    samples.append(dict(asset_name=name, objects=objects, active_seconds=None))
    per_image.append(dict(asset_name=name,predict_pipeline_seconds=predict_seconds,
        export_conversion_seconds=stamp()-start,stages_ms=result.speed,reached_max_det=False))
print('Đã tạo polygon cho',len(samples),'ảnh; ảnh không có dự đoán vẫn được giữ.')


In [ ]:
checkpoint_path = Path(model.ckpt_path)
checkpoint_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
bundle = dict(schema_version='1.0', task='instance_segmentation', name=(Path(CHECKPOINT).name + ' · segmentation')[:120],
    run=dict(model=CHECKPOINT, checkpoint=f'{checkpoint_path.name} sha256:{checkpoint_hash}',
        gpu=gpu, library_version=f'ultralytics=={ultralytics.__version__}; torch=={torch.__version__}',
        inference_seconds=sum(r['predict_pipeline_seconds'] for r in per_image)), samples=samples)
timing = dict(mask_representation='single polygon approximation; multipart/holes rejected', load_and_download_seconds=load_seconds, warmup_seconds=warmup_seconds,
    per_image=per_image, training_performed=False, manifest=names,
    image_sha256={name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in zip(names, paths)},
    checkpoint_sha256=checkpoint_hash, class_map=CLASS_MAP, confidence=CONFIDENCE,
    imgsz=IMGSZ, max_det=80, device=gpu,
    timing_definition='predict_pipeline includes image read/preprocess/forward/NMS; export conversion separate. Human and upload times excluded.')
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
publish_drive_result(ROOT, request, bundle, timing, environment)
print('Đã ghi kết quả lên Drive. Backend đang chạy sẽ tự nhập sau khi Drive đồng bộ.')
